# 🤖 Step 1: The Single-Agent "Copilot" Approach

## What we'll see in this notebook

We'll try to handle a production incident with a **single agent** — the way most people start.

You'll observe that a single agent:
- Gets confused juggling triage, diagnostics, AND remediation
- Has no access to infrastructure tools (can only guess)
- Produces generic advice instead of taking concrete action
- Can't verify its own fixes worked

This motivates **why** we need multi-agent orchestration.

---

## The Scenario

It's 2:41 AM. This alert just fired:

```
🔴 ALERT: High Latency on payment-api
   p95 latency: 2340ms (threshold: 2000ms)
   Duration: 5 minutes
   Impact: Customers experiencing slow checkouts
```

Let's see how a single agent handles it.

In [ ]:
import os
import json
from dotenv import load_dotenv
from agent_framework import ChatAgent
from agent_framework.azure import AzureAIAgentClient
from azure.identity.aio import AzureCliCredential

load_dotenv()

# Load our incident data
with open("data/incidents.json") as f:
    incidents = json.load(f)

alert = incidents[0]  # payment-api high latency
print(f"🔴 ALERT: {alert['title']}")
print(f"   Service: {alert['service']}")
print(f"   Severity: {alert['severity']}")
print(f"   Description: {alert['description']}")

In [ ]:
async def single_agent_approach():
    """Try to handle an incident with a single agent — no tools, no orchestration."""
    
    async with (
        AzureCliCredential() as credential,
        AzureAIAgentClient(async_credential=credential) as client,
    ):
        # A single "do everything" agent — the copilot approach
        copilot = client.create_agent(
            name="OnCallCopilot",
            instructions="""You are an on-call SRE assistant. When given an alert,
you should triage it, diagnose the root cause, suggest remediation steps,
and communicate the resolution. Handle the entire incident lifecycle.""",
        )

        # Feed it the alert
        prompt = f"""An alert just fired. Handle this incident end-to-end:

Alert: {alert['title']}
Service: {alert['service']}
Severity: {alert['severity']}
Description: {alert['description']}

Please: triage this, diagnose the root cause, fix it, and notify the team."""

        result = await copilot.run(prompt)
        print("\n🤖 Single Agent Response:\n")
        print(result.text)

await single_agent_approach()

## ❌ What's Wrong With This?

Look at the response above. The single agent:

1. **Guesses** instead of checking real metrics — it doesn't KNOW the CPU is at 45% or that pod-3 has 4 restarts
2. **Can't take action** — it says "restart the pod" but can't actually DO it
3. **Can't verify** — it says "check if latency improved" but has no way to check
4. **Generic advice** — the same response for any alert, no situational awareness
5. **No memory** — doesn't know this happened 2 days ago with the same root cause

### What we actually need:

| Capability | What's Missing |
|---|---|
| **Tools** | Agent can't query Prometheus, restart pods, or post to Slack |
| **Specialization** | One agent can't be expert at triage AND diagnostics AND remediation |
| **Orchestration** | No workflow: what happens if the fix fails? No retry logic, no escalation |
| **Memory** | No knowledge of past incidents or known issues |

---

## ➡️ Next Step

In **Step 2**, we'll decompose this into specialized agents, each with their own tools.

The architecture we're building:

```
Alert ──► Triage Agent ──► Diagnostics Agent ──► Remediation Agent ──► Verify ──► Comms Agent
          (history tool)    (metrics, logs)       (restart, scale)      (tests)    (slack, tickets)
```